## Appendix: Exploratory Data Analysis (detailed)

This is the project's full screening-stage EDA notebook — the original assignment-level
statistical exploration behind the concise summary in **2. Data Understanding** above.
It's included here in full because most of the project's later modelling decisions (which
macro features survive into Elastic Net, why certain lags are used, why the project treats
forecast-origin availability as an explicit assumption rather than a fact read off the
curated schema) trace back to a specific check somewhere in this notebook.

```{admonition} A dating note
:class: note
Sections 10 and 11 below screen candidate exogenous features "for SARIMAX." This notebook
predates the project's 2026-08-26 decision to fully remove SARIMAX (a root-stability
re-selection still validated ~9% worse real walk-forward RMSE than the reportable models —
see the project's model-comparison evidence). The screening itself wasn't wasted: the same
curated macro table and feature set feed Elastic Net today.
```

Statistical screens throughout are treated as indicative only — the sample is small and
several predictors have early missingness — not as confirmatory hypothesis tests.

| # | Section |
|---|---|
| 1 | Data Integrity Checks |
| 2 | Missing Values And Coverage |
| 3 | CPI Target Inspection (incl. seasonal decomposition & ACF/PACF) |
| 4 | External Variable Inspection |
| 5 | Stationarity Tests |
| 6 | Lead-Lag Correlation Analysis |
| 7 | Granger Causality Screening |
| 8 | Multicollinearity And VIF |
| 9 | Leakage And Forecast-Origin Availability Assumptions |
| 10 | Candidate Exogenous Feature Screening — headline |
| 11 | Candidate Exogenous Feature Screening — trimmed mean |
| 12 | Rolling Correlations / Relationship Stability |
| 13 | Raw Source Frequency Validation Spot Check |
| 14 | RBA Forecast Error Benchmark Preparation |
| 15 | Documented Limitations And Follow-Ups |

*Source: `notebooks/EDA.ipynb`*

In [1]:
from pathlib import Path
import re
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.platform_validation import validate_curated_with_pandera
from src.validation import validate_curated_dataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
plt.style.use('seaborn-v0_8-whitegrid')

DATA_PATH = ROOT / 'data' / 'curated' / 'quarterly_macro_features.csv'
AVAILABILITY_PATH = ROOT / 'data' / 'metadata' / 'series_availability.csv'
TARGET = 'cpi_yoy'
MAX_LAG = 4
SIGNAL_P_THRESHOLD = 0.10
SIGNAL_CORR_THRESHOLD = 0.20
MATERIAL_MISSING_THRESHOLD = 0.20
VIF_THRESHOLD = 10.0
CPI_FAMILY_BASE_COLS = [
    'cpi_qoq',
    'cpi_yoy',
    'trimmed_mean_cpi_qoq',
    'trimmed_mean_cpi_yoy',
]
CPI_DERIVED_DIAGNOSTIC_COLS = ['headline_trimmed_mean_yoy_gap']
CPI_DIAGNOSTIC_BASE_COLS = [
    'cpi_qoq',
    'trimmed_mean_cpi_qoq',
    'trimmed_mean_cpi_yoy',
    'headline_trimmed_mean_yoy_gap',
]
CPI_FAMILY_LAGGED_FEATURES = [
    'cpi_yoy_lag1',
    'cpi_yoy_lag4',
    'trimmed_mean_cpi_yoy_lag1',
    'trimmed_mean_cpi_yoy_lag4',
]